In [ ]:
! pip install bert-for-sequence-classification==0.0.4

In [ ]:
import os
import pandas as pd
import torch
import torch.optim as optim
import torch.nn as nn
import json

from transformers import AutoModel, AutoTokenizer

from bert_clf import BertCLF, train_evaluate, predict_metrics, prepare_data_notebook, prepare_dataset
from bert_clf.utils import set_global_seed

In [ ]:
# Import necessary modules
from bert_clf.src.early_stopping import EarlyStopping
import numpy as np

# Monkey-patch the EarlyStopping class
def _init__(self, config):  # Rename __init__ to _init__ to avoid recursion
    self.patience = config['training']['patience']
    self.counter = 0
    self.best_score = None
    self.early_stop = False
    self.val_loss_min = np.inf  # Use np.inf instead of np.Inf
    self.delta = config['training']['delta']
    self.config = config

# Assign the modified _init__ method to the original __init__ attribute
EarlyStopping.__init__ = _init__




### Prepare UC-UNSC dataset for testing

In [ ]:
df = pd.read_csv('sent.csv', sep =',')

In [ ]:
df

,Sentence_ID,Sentence,Arg_Type,Argument_IDs,Argument_Relations
0,1,"Once again, since the last briefing to the Cou...",claim,[1],{}
1,2,This is now the tenth time that the Council ha...,non-arg,[],{}
2,3,The General Assembly also took up the matter o...,non-arg,[],{}
3,4,"Following close to two weeks of relative calm,...",premise,[2],{}
4,5,The individuals involved called for secession ...,premise,[3],{}
...,...,...,...,...,...
4760,4761,We welcome Italy's decision to designate\ndial...,claim,[4556],{}
4761,4762,China supports practical and effective coopera...,claim,[4557],{}
4762,4763,We welcome all the positive efforts being made...,claim,[4558],{}
4763,4764,We hope that\nall the parties concerned will w...,claim,[4559],{}


In [ ]:
df = df[df['Arg_Type'] != 'non-arg']

In [ ]:
df['Label'] = df['Arg_Type'].apply(lambda x: 0 if x == 'premise' else 1)

<ipython-input-33-1f29226e7d61>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Label'] = df['Arg_Type'].apply(lambda x: 0 if x == 'premise' else 1)


In [ ]:
df['Label'].value_counts()

,count
Label,
1,2081
0,2024


### USElecDeb dataset

In [ ]:
df = pd.read_csv('sentence_db_candidate.csv', sep =',')

In [ ]:
df.shape

(29621, 18)

In [ ]:
valid = ['Claim', 'Premise']
df = df.loc[(df['Component'].isin(valid))]

In [ ]:
#splitting as the authors did
df_train = df[df['Set'] == 'TRAIN']
df_val = df[df['Set'] == 'VALIDATION']
df_test = df[df['Set'] == 'TEST']

df_train = df_train[['Speech', 'Component']]
df_val = df_val[['Speech', 'Component']]
df_test = df_test[['Speech', 'Component']]

In [ ]:
print(df_train.shape, df_val.shape, df_test.shape)

(10464, 2) (5241, 2) (6575, 2)


In [ ]:
df_test['Component'].value_counts()

,count
Component,
Claim,3361
Premise,3214


### Transformer Language Model

In [ ]:
config = dict(
    transformer_model = dict(
        model = "roberta-base",
        path_to_state_dict = False,
        device = 'cuda',
        dropout = 0.2,
        learning_rate = 2e-6,
        batch_size = 16,
        shuffle = True,
        maxlen = 128,
    ),
    data = dict(
        train_data_path = df_train,
        test_data_path = df_val,
        text_column = "Speech",
        target_column = "Component",
        random_state = 20,
        test_size = 0.3,
        stratify=True
    ),
    training = dict (
    save_state_dict = False, # if False the model will be saved using torch.save(<model_class>)
        # and should be loaded like this: model = torch.load()
        # you will have to install the library to do so
    early_stopping = True,
    delta = 0.001,
    patience = 7,
    num_epochs = 2,
    average_f1 = 'macro',
    other_metrics = ['micro', 'weighted'],
    output_dir = "../results/",
    class_weight = True
    )
)

In [ ]:
set_global_seed(seed=config['data']['random_state'])
os.makedirs(config['training']['output_dir'], exist_ok=True)

In [ ]:
device = torch.device(config['transformer_model']['device'])
tokenizer = AutoTokenizer.from_pretrained(
        pretrained_model_name_or_path=config['transformer_model']["model"]
    )
model_bert = AutoModel.from_pretrained(
    pretrained_model_name_or_path=config['transformer_model']["model"]
).to(device)

#for param in model_bert.parameters():
    #param.requires_grad = False

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
id2label, train_texts, valid_texts, train_targets, valid_targets = prepare_data_notebook(
    config=config, train_df = df_train, test_df = df_val
)

In [ ]:
model = BertCLF(
    pretrained_model=model_bert,
    tokenizer=tokenizer,
    id2label=id2label,
    dropout=config['transformer_model']['dropout'],
    device=device
    )

In [ ]:
model = model.to(device)

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=float(config['transformer_model']['learning_rate']))
criterion = nn.NLLLoss()

training_generator, valid_generator = prepare_dataset(
    tokenizer=tokenizer,
    train_texts=train_texts,
    train_targets=train_targets,
    valid_texts=valid_texts,
    valid_targets=valid_targets,
    config=config
)

In [ ]:
model = train_evaluate(
    model=model,
    training_generator=training_generator,
    valid_generator=valid_generator,
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=config['training']['num_epochs'],
    average=config['training']['average_f1'],
    config=config
)

==== Epoch 1 out of 2 ====


Evaluating loop: 100%|██████████| 328/328 [00:10<00:00, 31.79it/s]


Train F1: 0.5884095443204315
Eval F1: 0.672979239023552

Train F1 micro: 0.6178325688073395
Eval F1 micro: 0.6957782859078591

Train F1 weighted: 0.6054899409816441
Eval F1 weighted: 0.6929235181330172

==== Epoch 2 out of 2 ====


Evaluating loop: 100%|██████████| 328/328 [00:10<00:00, 31.81it/s]


Train F1: 0.7145657806534133
Eval F1: 0.6818961209292628

Train F1 micro: 0.7325114678899083
Eval F1 micro: 0.7052633807588076

Train F1 weighted: 0.7310330706506027
Eval F1 weighted: 0.6995527114508255




Computing final metrics...: 100%|██████████| 328/328 [00:08<00:00, 38.39it/s]


              precision    recall  f1-score   support

       Claim       0.70      0.81      0.75      2837
     Premise       0.72      0.59      0.65      2404

    accuracy                           0.71      5241
   macro avg       0.71      0.70      0.70      5241
weighted avg       0.71      0.71      0.70      5241



In [ ]:
model.to('cpu')

BertCLF(
  (pretrained_model): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm)

In [ ]:
# 4 min.
preds = []
for i,j in zip(df_test['Speech'], df_test['Component']):
    preds.append([model.predict(i), j, i])

In [ ]:
pred = []
for i in preds:
    pred.append(i[0])

true = []
for m in preds:
    true.append(m[1])

In [ ]:
from sklearn.metrics import classification_report
target_names = ['class 1', 'class 0']
print(classification_report(true, pred, target_names=target_names, digits=3))

              precision    recall  f1-score   support

     class 1      0.660     0.806     0.726      3361
     class 0      0.736     0.567     0.640      3214

    accuracy                          0.689      6575
   macro avg      0.698     0.686     0.683      6575
weighted avg      0.698     0.689     0.684      6575



### Testing on UC-UNSC

In [ ]:
preds = []
for i,j in zip(df['Sentence'], df['Label']):
    preds.append([model.predict(i), j, i])

In [ ]:
pred = []
for i in preds:
    pred.append(i[0])

true = []
for m in preds:
    true.append(m[1])

In [ ]:
p = []
for i in pred:
    if i == "Premise":
        p.append(0)
    if i == "Claim":
        p.append(1)

In [ ]:
from sklearn.metrics import classification_report
target_names = ['class 1', 'class 0']
print(classification_report(true, p, target_names=target_names, digits=3))

              precision    recall  f1-score   support

     class 1      0.747     0.690     0.717      2024
     class 0      0.719     0.772     0.745      2081

    accuracy                          0.732      4105
   macro avg      0.733     0.731     0.731      4105
weighted avg      0.733     0.732     0.731      4105



In [ ]:
import numpy as np
dummy = np.ones(4105)

In [ ]:
# baseline for task 2 on Ukraine data
target_names = ['class 0', 'class 1']
print(classification_report(true, dummy, target_names=target_names, digits=3))

              precision    recall  f1-score   support

     class 0      0.000     0.000     0.000      2024
     class 1      0.507     1.000     0.673      2081

    accuracy                          0.507      4105
   macro avg      0.253     0.500     0.336      4105
weighted avg      0.257     0.507     0.341      4105



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
